# 04 Delta Expert Optimizer

This loop goes beyond convex `G/A/S` mixing.  It uses `G` as the parent anchor
and learns how much of the aggregate delta and repair delta to inject:

```text
M = G + alpha(A - G) + beta(S - G)
```

The coefficients are independent for `body`, `head`, `router`, and each MoE
expert.  The purpose is to find a judge that can keep improving across rounds
instead of merely freezing when repeated repair drifts.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('/app/Object_Detection')
PROJECT = ROOT / 'dynamic_quality_aware_classwise_aggregation' / 'moe_dqa_judger'
OUT = PROJECT / 'output' / '04_delta_expert_optimizer'
OUT


## Run Loop


In [ ]:
import subprocess, sys

cmd = [
    sys.executable,
    str(PROJECT / 'scripts' / 'run_04_delta_expert_optimizer.py'),
    '--workspace-root', str(OUT),
    '--rounds', '1,2,3,4,5,6',
    '--mini-images', '512',
    '--random-candidates', '10',
    '--surrogate-iterations', '1',
    '--surrogate-pool', '72',
    '--surrogate-evals', '3',
    '--full-eval-topk', '2',
    '--val-batch-size', '32',
    '--notify-discord',
    '--force',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)


## Results


In [ ]:
best = pd.read_csv(OUT / 'stats' / '04_delta_expert_best_full.csv')
cols = [
    'round','candidate_id','map50','map50_95','precision','recall','score',
    'body_a','body_s','head_a','head_s','router_a','router_s',
    'expert0_a','expert0_s','expert1_a','expert1_s','expert2_a','expert2_s','expert3_a','expert3_s'
]
display(best[cols].head(20))
print((OUT / '04_delta_expert_optimizer_report.md').read_text())
